# Machine-Learning-Aufgabe: Herzerkrankungen aus PostgreSQL

Wende den Workflow aus Notebook 1 auf Daten an, die über fünf PostgreSQL-Tabellen verteilt sind.
Untersuch das Schema, bau den Join, prüf die Zielvariable und vergleich dann Klassifikatoren,
ohne den Testdatensatz für die Auswahl zu verwenden.

Die Datenbank und die SQL-Arbeit gehören zur Aufgabe. Behalt einzelne fehlende
Messwerte für die Imputation und erklär deine Entscheidungen zur Datenqualität.
Die Musterlösung folgt demselben Workflow; sie ist kein klinisch
validiertes Modell.

## Aufgabe

1. Untersuch die fünf Tabellen im Schema `heart` und verknüpf sie über den Patientenschlüssel.
2. Setz in jeder Messwerttabelle einen vorhandenen Eintrag voraus, prüf, ob es genau eine Zeile pro Person gibt,
   und behandle wiederholte Profile aus Eingaben und Label vor der Aufteilung.
3. Behalt `heart_attack` bei und leite `disease_present = 1 - heart_attack` ab.
   Schließ beide Zielspalten, die IDs und die Herkunftsmarkierungen von den Eingaben aus.
4. Leg vor der EDA stratifizierte Testdaten beiseite; erkunde nur die Trainingsdaten.
5. Bau eine Pipeline, die imputiert, kodiert, skaliert und eine logistische Regression trainiert.
   Fitte die Vorverarbeitung in jedem Trainings-Fold der CV separat.
6. Vergleich eine Baseline, Scaler und Hyperparameter, schreib dann einen Ansatz fest
   und evaluiere ihn einmal auf den Testdaten.

**Abgabe:** der SQL-Join, die Entscheidungen zur Datenqualität, der Validierungsvergleich,
der abschließende Classification Report und die Confusion Matrix mit Labels sowie eine kurze
Erklärung des Ergebnisses und seiner Grenzen.

## Zu beachten

- Wähl die positive Klasse und die Erfolgsmetrik vor dem Hyperparameter-Tuning. Die Accuracy allein kann wichtige Fehler verbergen.
- Nutz identische Folds und dieselbe Metrik, um Kandidaten zu vergleichen.
- Behalt fehlende Feature-Werte für einen Imputer, statt schwierige Testfälle stillschweigend zu entfernen.
- Numerische Kategoriecodes sind nicht unbedingt stetige Messwerte.
- Tune die Hyperparameter mit einem überschaubaren Randomized Search oder Grid Search, bevor du das Suchbudget erhöhst.
- Schließ nicht allein aus Korrelationen auf eine klinische Bedeutung oder die Herkunft der Daten.

## Datenüberblick

Das ist eine **Lehrfassung des Datensatzes UCI Cleveland Heart Disease**,
aufgeteilt auf fünf PostgreSQL-Tabellen. Eine verknüpfte Zeile steht für einen Patienteneintrag.
Die Aufgabe ist die retrospektive Klassifikation einer **dokumentierten Erkrankung**, nicht die Vorhersage
eines künftigen Herzinfarkts. Der Datensatz ist klein und historisch; das ist kein Tool für medizinische
Entscheidungen. [Originaldokumentation von UCI](https://archive.ics.uci.edu/dataset/45/heart+disease).

### Zielvariable: dem Spaltennamen nicht trauen

Das Feld `heart_attack` der Datenbank hat eine **umgekehrte binäre Kodierung** im Vergleich
zum ursprünglichen UCI-Label für eine vorhandene Krankheit:

| `heart_attack` in der Datenbank | Abgeleitetes `disease_present` | Bedeutung |
|---|---|---|
| 1 | 0 | Laut Definition der Quelle keine Krankheit dokumentiert |
| 0 | 1 | Krankheit vorhanden |

Behalt das ursprüngliche Feld bei, prüf, dass es nur 0 und 1 enthält, und leite
`disease_present = 1 - heart_attack` ab. Nutz **nur `disease_present` als
Zielvariable für die Modellierung**. Klasse 0 bedeutet nicht allgemein gesund.
Keine der beiden Zielspalten gehört zu den Features.

Diese Zuordnung wurde am 2026-10-08 mit UCI abgeglichen: Alle 303 Einträge mit den
nötigen verknüpften Tabellen stimmen nach Berücksichtigung der umkodierten Kategorien und fehlender
Messwerte überein. Eine höhere Herzfrequenz oder weniger Angina in Klasse 1 der Datenbank
deutet deshalb nicht auf einen umgekehrten klinischen Zusammenhang hin: Diese Klasse bedeutet
**keine Krankheit dokumentiert**. Nutz die dokumentierte Kodierung, nicht Korrelationen,
um die Labels zu interpretieren.

### Feature-Wörterbuch: Datenbankcodes, nicht die ursprünglichen UCI-Codes

Ein EKG zeichnet die elektrische Aktivität des Herzens auf. Angina bezeichnet Brustschmerzen,
die mit einer verminderten Durchblutung des Herzens einhergehen; die ST-Strecke ist ein Teil
des EKGs. Die numerischen Codes unten sind Kennungen von Kategorien, keine Messwerte.

| Datenbankspalte | Bedeutung und Kodierung | Behandlung |
|---|---|---|
| `patient_id` | Patientenschlüssel aus dem Join | Auf Eindeutigkeit prüfen; von den Eingaben ausschließen |
| `age` | Alter in Jahren | Numerisch |
| `sex` | Erfasstes Geschlecht: 0 = weiblich, 1 = männlich | Kategorial |
| `chest_pain_type` | 0 = asymptomatisch, 1 = atypische Angina, 2 = nicht anginöser Schmerz, 3 = typische Angina | Kategorial |
| `resting_blood_pressure` | Ruheblutdruck bei der Aufnahme, mm Hg | Numerisch |
| `serum_cholestoral` | Serumcholesterin, mg/dL; die Schreibweise folgt dem Schema | Numerisch |
| `fasting_blood_sugar` | Über 120 mg/dL: 0 = falsch, 1 = wahr | Kategorial |
| `resting_electrocardiographic_results` | 0 = wahrscheinliche/sichere linksventrikuläre Hypertrophie, 1 = normal, 2 = ST-T-Veränderung | Kategorial |
| `maximum_heartrate_achieved` | Maximale Herzfrequenz unter Belastung, Schläge/Minute | Numerisch |
| `exercise_induced_angina` | 0 = nein, 1 = ja | Kategorial |
| `oldpeak` | Belastungsinduzierte ST-Senkung im Vergleich zur Ruhe | Numerisch |
| `slope_of_the_peak_exercise_st_segment` | 0 = absteigend, 1 = flach, 2 = ansteigend | Kategorial |
| `number_of_major_vessels_colored_by_flourosopy` | Gefäßanzahl 0 bis 3; Code 4 steht für fehlend | Numerische Anzahl; 4 durch fehlend ersetzen |
| `thal` | Testergebnis: 1 = fixierter Defekt, 2 = normal, 3 = reversibler Defekt; Code 0 steht für fehlend | Kategorial; 0 durch fehlend ersetzen |
| `real_data` | Herkunftsmarkierung der Lehrdaten, manchmal nicht vorhanden | Nur für die Qualitätsprüfung; von den Eingaben ausschließen |
| `heart_attack` | Ursprüngliches, invertiertes binäres Feld | Zur Kontrolle beibehalten; von den Eingaben ausschließen |
| `disease_present` | Abgeleitetes Label: 0 = keine Krankheit dokumentiert, 1 = Krankheit vorhanden | Nur Zielvariable |

Die Datenbankcodes unterscheiden sich von den ursprünglichen UCI-Codes; **misch die beiden
Wörterbücher nicht**. Behalt die Kategorien der Datenbank unverändert bei und kodiere sie
per One-Hot Encoding. Wandle die beiden Platzhaltercodes für fehlende Werte vor der Imputation um.

Warum diese 13 Eingaben? Damit bleiben die diagnostischen Benchmark-Attribute der Quelle erhalten.
Die Gefäßanzahl und `thal` hängen von speziellen Tests ab, und mehrere andere Features
stammen aus einem Belastungstest. Bei einem früheren Screening lägen sie nicht alle
vor. Eingaben auszuwählen heißt zu wissen, was zum Zeitpunkt der Vorhersage
verfügbar ist, und nicht einfach jede Spalte der Datenbank zu verwenden.

### Was ist anders als in der Originaldatei?

Die Überprüfung ergab 303 verknüpfte Einträge: 264 stimmen nach der Umkodierung mit allen
13 UCI-Messwerten überein, 39 stimmen mit den verfügbaren Messwerten überein, haben aber zusätzlich
fehlende Werte bei Alter, Geschlecht oder maximaler Herzfrequenz. Ein ursprüngliches Profil aus Eingaben und Label
kommt unter zwei IDs doppelt vor, während ein anderer ursprünglicher Eintrag fehlt.

Der frühere Left Join behielt außerdem 50 zusätzliche Patientinnen und Patienten ohne die nötigen
Einträge in den Messwerttabellen. Unterscheide **einen fehlenden verknüpften Eintrag** von
**einem fehlenden Messwert in einem vorhandenen Eintrag**: Schließ den ersten Fall aus, behalt aber
den zweiten für die auf den Trainingsdaten gefittete Imputation. Nach dem Entfernen des wiederholten Profils
verwendet die aktuelle Lösung 302 Einträge. Diese Zahlen beschreiben die überprüfte
Lehrdatenbank, keinen garantierten künftigen Zustand der Datenbank.

## Import

In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text

# Imports für die Modellierung hinzufügen, sobald du sie verwendest.

## Daten beschaffen

Das Schema `heart` enthält `patient`, `chest_pain`, `blood_metrics`,
`pressure_vessels_angina` und `heart_attack_history`.
Untersuch die Tabellen in DBeaver und schreib dann den Join selbst.
Nutz `p.id = other_table.patient_id`, wähl die Spalten explizit aus und gib dem
Patientenschlüssel den Alias `patient_id`.

Nutz Inner Joins für die Messwerttabellen, um einen vorhandenen Eintrag vorauszusetzen,
statt Patientinnen und Patienten ganz ohne Messwerte zu behalten. Ein Inner Join entfernt
fehlende Werte innerhalb einer vorhandenen Zeile **nicht**: Behalt sie für die Imputation.
Schließ die eindeutig erfundene Herkunftsmarkierung aus, die in der Musterlösung zu sehen ist;
eine fehlende Herkunftsmarkierung allein ist kein Grund, einen Eintrag zu verwerfen.

Kopiere `.env.example` nach `.env` und trag die bereitgestellten Zugangsdaten ein.
Der Code baut die Verbindungs-URL aus diesen Feldern, gib das Passwort also
unverändert ein, ohne URL-Kodierung. Ein vorhandener vollständiger `DB_STRING` wird ebenfalls unterstützt.
Zugangsdaten dürfen nie per Commit ins Repo gelangen.

Die nächste Abfrage ist eine **Vorschau auf zehn Patientinnen und Patienten**, kein Datensatz für die Modellierung.
Ersetz sie durch deinen vollständigen Join der fünf Tabellen, bevor du weitermachst.
Die Verbindung ist schreibgeschützt und wird nach dem Lesen geschlossen; der DataFrame bleibt verfügbar.

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "solutions":
    PROJECT_ROOT = PROJECT_ROOT.parent
load_dotenv(PROJECT_ROOT / ".env")

DB_STRING = os.getenv("DB_STRING")
if not DB_STRING or "<" in DB_STRING:
    # URL aus den Feldern in .env bauen; URL.create behandelt Sonderzeichen.
    field_names = ("USER_DB", "PASSWORD", "HOST", "DATABASE")
    fields = {name: os.getenv(name) for name in field_names}
    if any(not value or "<" in value for value in fields.values()):
        raise ValueError(
            "Set DB_STRING or USER_DB, PASSWORD, HOST, DATABASE and PORT in .env."
        )
    try:
        port = int(os.getenv("PORT") or "5432")
    except ValueError:
        raise ValueError("PORT in .env must be an integer.") from None
    DB_STRING = URL.create(
        "postgresql+psycopg2",
        username=fields["USER_DB"],
        password=fields["PASSWORD"],
        host=fields["HOST"],
        port=port,
        database=fields["DATABASE"],
    )

db = create_engine(DB_STRING, connect_args={"connect_timeout": 10})

In [ ]:
# Nur eine Vorschau: vor der Modellierung durch deinen Join der fünf Tabellen ersetzen.
query_string = "SELECT * FROM heart.patient LIMIT 10"
try:
    with db.connect() as connection, connection.begin():
        connection.execute(text("SET TRANSACTION READ ONLY"))
        df_sqlalchemy = pd.read_sql(text(query_string), connection)
finally:
    db.dispose()
df_sqlalchemy.head()

In [ ]:
# Optional: den fertigen Join lokal in der von Git ignorierten Datei speichern.
# df_sqlalchemy.to_csv(PROJECT_ROOT / "data" / "heart_data.csv", index=False)

## 1. Deinen fertigen Join prüfen

Ersetz die Vorschau durch deinen Join. Prüf die Zeilenanzahl, fehlende oder wiederholte
Patientenschlüssel und die Codes der Zielvariable. Verwirf widersprüchliche Zeilen
für dieselbe Person nicht stillschweigend: Korrigiere den Join.

Zähl exakte Duplikate von **Profilen aus Eingaben und Label** und lass `patient_id`
und `real_data` bei diesem Vergleich weg. Die aktuelle Datenbank enthält ein wiederholtes
Profil unter verschiedenen IDs. Entfern dieses wiederholte Profil vor der Aufteilung,
damit dieselbe vollständige Beobachtung nicht auf beiden Seiten der Evaluation auftauchen kann.
Identische Profile beweisen nicht, dass zwei Einträge dieselbe reale Person bezeichnen;
es handelt sich um eine dokumentierte Bereinigung der Lehrdaten.

Wandle Code 4 der Gefäßanzahl und Code 0 von `thal` in fehlende Werte um. Nutz
diese Platzhaltercodes nicht als Messwerte oder Kategorien. Behalt gewöhnliche fehlende
Eingaben für den Imputer der Pipeline.

**Erwartetes Ergebnis:** ein verknüpfter DataFrame mit einer Zeile pro infrage kommender Person,
ein Bericht über die Ausschlüsse sowie die ursprüngliche und die korrekt abgeleitete Zielvariable.
Derzeit liefert der Join 303 Zeilen, und nach der Deduplizierung der Profile bleiben 302;
untersuch dein tatsächliches Ergebnis, statt diese Zahlen zu erzwingen.

In [ ]:
# Optional: den fertigen Join lokal in der von Git ignorierten Datei speichern.
# df_sqlalchemy.to_csv(PROJECT_ROOT / "data" / "heart_data.csv", index=False)

## 2. Zielvariable, Metrik und Aufteilung festlegen

Behalt `heart_attack` bei, validiere seine binären Codes und leite
`disease_present = 1 - heart_attack` ab. **Klasse 1 bedeutet jetzt Krankheit vorhanden.**
Lass `heart_attack`, `disease_present`, `patient_id` und `real_data`
aus `X` heraus; nutz nur `disease_present` als `y`.

Wähl F2 für Klasse 1 als wichtigste Auswahlmetrik, neben Precision,
Recall und Accuracy. Ein False Negative übersieht einen Eintrag mit vorhandener Krankheit;
ein False Positive markiert einen Eintrag, der laut Definition der Quelle keine Krankheit hat.

$$F_2 = \frac{5TP}{5TP + FP + 4FN}$$

F2 betont den Recall; der Koeffizient 4 legt keine echten klinischen Kosten fest.
Eine konstant positive Baseline hat einen Recall von 1, löst aber viele Fehlalarme aus.
Nutz `pos_label=1` durchgängig.

Leg einen stratifizierten Testdatensatz beiseite. Nutz wie in Notebook 1 eine CV nur auf den Trainingsdaten, oder füg
wie in der Musterlösung einen separaten Validierungsdatensatz hinzu. Die Vorverarbeitung gehört
in jeden Trainings-Fold der CV.

<img src="assets/heart-evaluation-protocol.svg" width="650" style="max-width: 100%;" alt="Kreuzvalidierung nur auf den Trainingsdaten, ein separater Validierungsdatensatz und eine abschließende Evaluation auf den Testdaten.">

**Erwartetes Ergebnis:** benannte Teilmengen, ihre Größen, die Klassenanteile und ein Plan für die
Evaluation. Lass die Testdaten bis Schritt 6 unberührt.
[Referenz zur Metrik](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.fbeta_score.html).

In [ ]:
# Optional: den fertigen Join lokal in der von Git ignorierten Datei speichern.
# df_sqlalchemy.to_csv(PROJECT_ROOT / "data" / "heart_data.csv", index=False)

## 3. Trainingsdaten erkunden

Untersuch Datentypen, das Fehlen von Werten, die Klassenverteilung, Kategoriecodes und numerische Wertebereiche
nur in den Trainingsdaten. Stell die maximale Herzfrequenz unter Belastung nach `disease_present` dar
und vergleich in jeder Klasse den Anteil mit belastungsinduzierter Angina.
Kennzeichne Klasse 0 als "No disease recorded" und Klasse 1 als "Disease present".

Diese Plots greifen die ursprüngliche Verwirrung über gesünder aussehende Einträge der Klasse 1
auf: Das war die **rohe invertierte Zielvariable**, nicht die korrigierte.
Interpretiere die tatsächlichen Ausgaben, ohne eine Korrelation als klinischen Beleg zu werten.

**Erwartetes Ergebnis:** eine knappe Zusammenfassung der Datenqualität und numerische/kategoriale
Feature-Gruppen mit Begründung für ihre Behandlung. Lösch einen extremen
Messwert nicht nur deshalb, weil er ungewöhnlich aussieht.

In [ ]:
# Optional: den fertigen Join lokal in der von Git ignorierten Datei speichern.
# df_sqlalchemy.to_csv(PROJECT_ROOT / "data" / "heart_data.csv", index=False)

## 4. Baseline festlegen und Pipeline aufbauen

Wähl eine einfache Baseline, die zu deiner Metrik passt, und bau dann einen `ColumnTransformer` und eine `Pipeline` mit Imputation, kategorialer Kodierung, einem numerischen Scaler und logistischer Regression. Nimm bei kodierten Kategorien wie der Art des Brustschmerzes nicht an, dass die ganzen Zahlen Messwerte mit gleichen Abständen darstellen.

**Erwartetes Ergebnis:** eine Baseline und eine vollständige Pipeline, evaluiert nach deinem Protokoll für die Entwicklungsdaten. Erklär, was die Baseline vorhersagt; ein hoher Recall allein kann viele False Positives verbergen.

In [ ]:
# Optional: den fertigen Join lokal in der von Git ignorierten Datei speichern.
# df_sqlalchemy.to_csv(PROJECT_ROOT / "data" / "heart_data.csv", index=False)

## 5. Kandidaten vergleichen und auswählen

Vergleich unskalierte, standardisierte und Min-Max-skalierte Pipelines mit derselben Metrik und denselben CV-Folds. Führ auf den Trainings-Folds eine überschaubare Suche durch; wenn du einen separaten Validierungsdatensatz hast, nutz ihn für den abschließenden Vergleich der Kandidaten. Nimm die Baseline auf und lass die Test-Scores aus dieser Tabelle heraus.

**Erwartetes Ergebnis:** eine Vergleichstabelle und eine ausgewählte Konfiguration mit kurzer Begründung. Gib an, wenn das Hyperparameter-Tuning einen einfacheren Kandidaten nicht übertrifft.

Setz bei beiden Suchen `verbose=1`, damit sie anzeigen, dass sie laufen.
Fang mit einem kleinen Budget an, etwa 10 zufälligen Kandidaten und 8 Kandidaten im Netz.

In [ ]:
# Optional: den fertigen Join lokal in der von Git ignorierten Datei speichern.
# df_sqlalchemy.to_csv(PROJECT_ROOT / "data" / "heart_data.csv", index=False)

## 6. Einmal auf dem Testdatensatz evaluieren

Beantworte zuerst die Fragen am Ende. Schreib die ausgewählte Konfiguration fest,
trainiere sie auf allen Entwicklungsdaten neu und evaluiere einmal auf den Testdaten.
Eine CV-Suche mit `refit=True` trainiert auf den Daten neu, die an `fit` übergeben werden;
eine separate Validierungsteilmenge wird nicht automatisch einbezogen.

**Erwartetes Ergebnis:** F2, Precision, Recall, Accuracy, ein Classification Report
und eine Confusion Matrix mit den Labels "No disease recorded" / "Disease present".
Erklär False Positives und False Negatives anhand von `disease_present`, nicht
anhand der rohen Datenbankspalte. Nutz das Ergebnis nicht für ein erneutes Hyperparameter-Tuning.

In [ ]:
# Optional: den fertigen Join lokal in der von Git ignorierten Datei speichern.
# df_sqlalchemy.to_csv(PROJECT_ROOT / "data" / "heart_data.csv", index=False)

## 7. Dein Ergebnis erklären

Fass zusammen, ob das ausgewählte Modell die Baseline bei deiner gewählten Metrik übertrifft, was die Fehler bedeuten, und nenn zwei Grenzen der Daten oder der Evaluation. Unterscheide die Vorhersage der Labels im Datensatz vom künftigen Herzinfarktrisiko.

**Erwartetes Ergebnis:** ein kurzes Fazit zu deinem reproduzierbaren Notebook. Vergleich danach deine Entscheidungen mit der [Musterlösung](solutions/2_exercise_machine_learning.ipynb); unterschiedliche gültige Entscheidungen müssen nicht zu identischen Scores führen.

## Prüfe dein Verständnis

Erklär vor der abschließenden Evaluation auf den Testdaten:

1. Warum `patient_id` ausschließen, auch wenn die ID die Validierungs-Scores verbessert?
2. Warum den Imputer in jedem Trainings-Fold der CV fitten?
3. Warum Modelle nicht anhand ihrer Test-Scores auswählen?
4. Warum `heart_attack` beibehalten, aber nie an das korrigierte Modell übergeben?

Erklär deine Überlegungen, bevor du in die Musterlösung schaust.